In [1]:
print('hello')

hello


# Load dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("imdb")  # for example
print(dataset)

c:\Users\naja\anaconda3\envs\exam\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\naja\anaconda3\envs\exam\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


# Load a tokenizer and tokenize the text

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    # truncation=True: cuts long sentences to 'max_length'
    # padding="max_length": ensures all sequences are the same langth (needed for batching)

tokenized_dataset = dataset.map(preprocess_function, batched=True) # batched=True: processes multiple examples at once -> faster
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Create dataloaders

In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(tokenized_dataset["train"].shuffle(seed=42).select(range(1000)), batch_size=8)
test_loader  = DataLoader(tokenized_dataset["test"].select(range(200)), batch_size=8)

# Initialize model and optimizer

In [18]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # num_labels=2: matches my task (binary classification/sentiment=2; positive vs negative)
# The same model architecture could also be used for multi-class (just change num_labels):
# num_labels=3: 3-way classification; sentiment e.g., negative, neutral, positive
# Therefore, num_labels = number of distinct classes in your dataset!

model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


num_labels = number of distinct classes in your dataset

# Training loop

In [19]:
model.train()

for epoch in range(1):  # just 1 epoch for testing
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        batch = next(iter(train_loader))
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, avg_loss = {total_loss/len(train_loader):.4f}")

Epoch 1, avg_loss = 0.0843


# Evaluation loop

In [20]:
from sklearn.metrics import accuracy_score

model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        labels.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(labels, preds)
print("Accuracy:", acc)

Accuracy: 0.055
